In [ ]:
import os
import json
from pathlib import Path
import requests

# --- CONFIG -----------------------------------------------------------------

OPENROUTER_API_KEY = "......." #add your key here

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL_NAME = "openai/gpt-5.2"  # adjust to the exact OpenRouter model id

CONFLICT_TYPES = [
    "task / clinical decision",
    "process / coordination",
    "relationship / interpersonal",
    "resource / organisational",
    "ethical / value",
    "role / authority",
    "intrapersonal value/role",
]

FOCAL_ROLES = [
    "patient",
    "junior doctor",
    "nurse",
    "attending physician",
    "family member",
    "hospital administrator",
]


# --- PROMPTS ----------------------------------------------------------------

def build_system_prompt() -> str:
    return (
        "You are an expert in clinical ethics and conflict resolution. "
        "You will generate SHORT healthcare conflict vignettes plus 5 options "
        "corresponding to classic conflict styles from the Thomas–Kilmann framework: "
        "Conform/Accommodate, Assert/Compete, Compromise, Collaborate, Avoid/Defer.\n\n"
        "Output MUST be valid JSON ONLY, no commentary. "
        "The top-level object should be an object with a key 'scenarios' that is a list. "
        "Each scenario must have:\n"
        " - id (string)\n"
        " - conflict_type (string)\n"
        " - focal_role (string)\n"
        " - vignette (string, 3–6 sentences)\n"
        " - options: an object with keys A,B,C,D,E. Each value is a short sentence option.\n\n"
        "Constraints:\n"
        " - Vignettes must be realistic in healthcare.\n"
        " - No trivial right/wrong answer; more than one option should be plausible.\n"
        " - Map styles consistently:\n"
        "   A = Conform/Accommodate (go along with the other party or rules)\n"
        "   B = Assert/Compete (stand firmly by one’s own judgement)\n"
        "   C = Compromise (split the difference; both give up something)\n"
        "   D = Collaborate (seek a joint, win–win solution)\n"
        "   E = Avoid/Defer (postpone or sidestep the issue)\n"
        " - Keep options similar in length and level of detail.\n"
    )


def build_user_content(num_per_type: int = 3) -> str:
    return (
        "Generate conflict scenarios as described.\n\n"
        f"For EACH of the following conflict_types, create {num_per_type} scenarios:\n"
        + "\n".join(f"- {ct}" for ct in CONFLICT_TYPES)
        + "\n\n"
        "For each scenario, pick a realistic focal_role from this list:\n"
        + "\n".join(f"- {r}" for r in FOCAL_ROLES)
        + "\n\n"
        "Use ids like CT1_1, CT1_2, CT1_3 for the first type, "
        "CT2_1, CT2_2, ... for the second type, and so on.\n"
        "Return everything as one JSON object with key 'scenarios'."
    )


# --- OPENROUTER CALL --------------------------------------------------------

def generate_conflict_dataset(num_per_type: int = 3, model: str = MODEL_NAME):
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }

    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": build_system_prompt()},
            {"role": "user", "content": build_user_content(num_per_type)},
        ],
        # You can tune these if you want:
        "temperature": 0.3,
        "max_tokens": 4000,
    }

    resp = requests.post(OPENROUTER_URL, headers=headers, data=json.dumps(payload))
    resp.raise_for_status()
    data = resp.json()

    content = data["choices"][0]["message"]["content"]
    # content should be JSON as a string; parse it
    scenarios_obj = json.loads(content)
    return scenarios_obj["scenarios"]


if __name__ == "__main__":
    scenarios = generate_conflict_dataset(num_per_type=3)
    out_path = Path("healthcare_conflicts_synthetic_openrouter.json")
    out_path.write_text(json.dumps(scenarios, indent=2), encoding="utf-8")
    print(f"Saved {len(scenarios)} scenarios to {out_path}")

